# Final Report — VisionServeX v2.46.0

This notebook is the **read-only consumer** of artifacts that the rest of
`RUN_ALL.ipynb` (or `run_all.sh`) generated. It must never overwrite the
reconciled ledger and must never read the legacy 11-column static ledger
under `archive_legacy/`.

Inputs (all produced earlier in the same run):

* `99_final_report/reports/model_coverage_ledger.csv` — reconciled, detailed.
* `99_final_report/reports/final_winners.json` — computed from current-run leaderboards.
* `99_final_report/reports/benchmark_notebook_coverage_audit.json` — coverage audit result.
* `99_final_report/reports/generated_artifact_integrity.json` — provenance.

Every count, leaderboard, and remaining-blocker row in this notebook is
computed from those files. There are no hardcoded numbers.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd

NB_ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path('/home/arash/PycharmProjects/VisionServeX/notebook')
TASK = NB_ROOT / '99_final_report'
REPORTS = TASK / 'reports'

import visionservex
print(f'VisionServeX {visionservex.__version__}')

LEDGER_CSV = REPORTS / 'model_coverage_ledger.csv'
LEDGER_JSON = REPORTS / 'model_coverage_ledger.json'
WINNERS_JSON = REPORTS / 'final_winners.json'
COVERAGE_JSON = REPORTS / 'benchmark_notebook_coverage_audit.json'

assert LEDGER_CSV.exists(), f'Reconciled ledger missing at {LEDGER_CSV}. Run RUN_ALL.ipynb (do not run Final_Report.ipynb alone).'
assert LEDGER_JSON.exists(), f'Reconciled JSON ledger missing at {LEDGER_JSON}.'
print(f'Reading ledger: {LEDGER_CSV.resolve()}')

In [ ]:
# Hard schema validation — fail if the static 11-column legacy schema sneaks back in.
df = pd.read_csv(LEDGER_CSV)

OLD_SCHEMA_COLS = {
    'model_id', 'family', 'task', 'engine', 'license_status', 'default_safe',
    'install_extra', 'implementation_status', 'final_state', 'blocker_code', 'run_mode',
}
REQUIRED_COLS = {
    'model_id', 'final_state', 'blocker_code', 'blocker_category', 'evidence_artifact',
    'evidence_source', 'called_in_current_notebook_run', 'current_run_id',
    'current_run_artifact_exists', 'metric_origin',
}

current_cols = set(df.columns)
old_schema_detected = current_cols == OLD_SCHEMA_COLS
assert not old_schema_detected, 'OLD 11-COLUMN STATIC SCHEMA DETECTED — RUN_ALL.ipynb is broken; do not consume this ledger.'
missing_required = REQUIRED_COLS - current_cols
assert not missing_required, f'Reconciled ledger missing required columns: {sorted(missing_required)}'

print('LEDGER SCHEMA VERIFICATION')
print(f'  rows: {len(df)}')
print(f'  columns: {len(df.columns)}')
print(f'  old_schema_detected: {old_schema_detected}')
print(f'  required_columns_present: {len(missing_required) == 0}')
if 'current_run_id' in df.columns:
    valid = df['current_run_id'].dropna()
    print(f'  current_run_id sample: {valid.iloc[0] if len(valid) else None!r}')

In [ ]:
# Final state counts — purely derived from the reconciled ledger.
state_counts = df['final_state'].fillna('(missing)').value_counts(dropna=False)
print('final_state counts:')
for state, n in state_counts.items():
    print(f'  {state:32s}: {n}')

# Health summary
HEALTHY_STATES = {'smoke_passed', 'contract_passed', 'benchmark_passed', 'wired', 'partial'}
is_healthy = df['final_state'].isin(HEALTHY_STATES)
n_healthy = int(is_healthy.sum())
n_total = len(df)
n_non_healthy = n_total - n_healthy
print()
print('HEALTH SUMMARY (derived):')
print(f'  total            : {n_total}')
print(f'  healthy          : {n_healthy}')
print(f'  non_healthy      : {n_non_healthy}')

In [ ]:
# Final winners — read from the JSON the reconciler computed.
if WINNERS_JSON.exists():
    winners = json.loads(WINNERS_JSON.read_text())
else:
    winners = {}
print('Final winners (read from final_winners.json):')
for k, v in winners.items():
    if isinstance(v, (str, int, float)) or v is None:
        print(f'  {k:48s}: {v}')

In [ ]:
# Notebook benchmark-coverage audit — generated by RUN_ALL.
if COVERAGE_JSON.exists():
    audit = json.loads(COVERAGE_JSON.read_text())
    counts = audit.get('missing_counts') or {}
    print('Notebook benchmark-coverage audit (read from benchmark_notebook_coverage_audit.json):')
    for k, v in counts.items():
        print(f'  {k:48s}: {v}')
    if any(v for v in counts.values()):
        print('  WARNING: at least one missing-from-notebook count is non-zero.')
else:
    print('benchmark_notebook_coverage_audit.json missing — run RUN_ALL.ipynb to generate it.')

In [ ]:
# Remaining non-healthy table — derived from the reconciled ledger.
non_healthy_df = df[~df['final_state'].isin(HEALTHY_STATES)].copy()
cols = [c for c in ['model_id', 'task', 'final_state', 'blocker_code', 'blocker_category'] if c in non_healthy_df.columns]
print(f'Remaining non-healthy rows: {len(non_healthy_df)}')
if len(non_healthy_df):
    print(non_healthy_df[cols].to_string(index=False))

In [ ]:
# Cross-check: text counts (this notebook) must equal CSV value counts.
text_total = n_total
text_healthy = n_healthy
text_non_healthy = n_non_healthy
csv_total = len(df)
csv_healthy = int(df['final_state'].isin(HEALTHY_STATES).sum())
csv_non_healthy = csv_total - csv_healthy
assert text_total == csv_total
assert text_healthy == csv_healthy
assert text_non_healthy == csv_non_healthy
print(f'CROSS-CHECK OK: text counts match CSV counts ({csv_healthy}/{csv_total} healthy).')

# Also reject the archive_legacy ledger path explicitly.
ARCHIVE_LEGACY = NB_ROOT / 'archive_legacy/outputs/visionservex_master_outputs/final/model_coverage_ledger.csv'
if ARCHIVE_LEGACY.exists():
    print(f'NOTE: archive_legacy ledger exists at {ARCHIVE_LEGACY} but is NOT read by this notebook.')

print('Final report complete (read-only consumer).')